# Feature decisions

Two questions, both about which columns reach the model.

`Attr21` dominates the boosting model. Its missingness was already recorded in the feature
analysis as possibly an artefact of how the file was assembled, and the indicator was left out
of the model for that reason — but excluding the *indicator* does not remove the signal.
LightGBM routes NaN natively and rebuilds the flag from the column itself.

The same mechanism raises a second question: six missingness indicators are built for both
branches, and the boosting branch may not need any of them.

Everything below runs on the training split alone — 4387 rows, 306 positives. Configurations are
compared fold by fold on the same 25 folds. The holdout is untouched.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.config import DATA_5YEAR_PATH
from src.data import split_holdout
from src.pipelines import make_boosting_pipeline, make_linear_pipeline
from src.train import compare_pipelines, cross_validate_model

X_train, X_holdout, y_train, y_holdout = split_holdout(DATA_5YEAR_PATH)
feature_cols = X_train.columns.tolist()


def boosting(cols):
    return make_boosting_pipeline(LGBMClassifier(verbose=-1), cols)


def linear(cols):
    return make_linear_pipeline(
        LogisticRegression(class_weight="balanced", max_iter=1000), cols
    )


def report(diff: np.ndarray) -> str:
    """Mean difference with twice its standard error."""
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    return f"{diff.mean():+.4f} ± {2 * se:.4f}"


def show(diff: dict[str, np.ndarray]) -> None:
    """Print one line per metric: paired difference and how often it held."""
    for name, d in diff.items():
        print(f"{name:15s} {report(d)}   folds in favour: {(d > 0).sum()} / {len(d)}")

## Reference points

Both branches on all 64 columns, default hyperparameters, nothing done about the imbalance yet.
These are not results; they are what every comparison below is measured against.

Boosting reaches 0.817 PR-AUC where random ranking gives 0.070 and the logistic baseline gives
0.348 ± 0.039. A gap that wide is plausible on this data — the 64 ratios are non-monotone with
tails hundreds of times the 99th percentile, which cripples a linear model and costs a tree
nothing — but it is also wide enough to be worth checking before it is believed.

In [2]:
for name, pipe in [
    ("boosting", boosting(feature_cols)),
    ("linear", linear(feature_cols)),
]:
    res = cross_validate_model(pipe, X_train, y_train)
    print(
        f"{name:9s} PR-AUC {res['pr_auc'].mean():.3f} ± {res['pr_auc'].std():.3f}"
        f"   P@3% {res['precision_at_k'].mean():.3f}"
    )

boosting  PR-AUC 0.817 ± 0.043   P@3% 0.977
linear    PR-AUC 0.556 ± 0.045   P@3% 0.717


## Where the boosting score comes from

Split counts: how often each column was chosen for a split across the ensemble. With thirteen
pairs correlated above 0.99, these numbers are unstable inside a correlated group and cannot
carry a conclusion. They can say where to look.

`Attr21` leads. Its missingness is exactly the pattern the feature analysis could not attribute:
80 rows, 78 of them bankrupt. The indicator was kept out of `MISSING_INDICATOR_COLS`, but that
decision only binds the linear branch — a tree sends NaN down whichever side of a split improves
the loss, so it reconstructs the flag without being handed one.

In [3]:
pipe = boosting(feature_cols)
pipe.fit(X_train, y_train)

pd.Series(
    pipe.named_steps["model"].feature_importances_,
    index=pipe.named_steps["prep"].get_feature_names_out(),
).sort_values(ascending=False).head(10)

raw__Attr21    230
raw__Attr27    209
raw__Attr34    179
raw__Attr24    167
raw__Attr46    160
raw__Attr58    121
raw__Attr45     83
raw__Attr40     81
raw__Attr56     79
raw__Attr35     67
dtype: int32

## What the column is worth to boosting

Both configurations are cross-validated on the same folds, so the fold-to-fold spread — around
0.042, and the dominant source of variance — cancels, leaving the effect of the change. Reported
as the mean paired difference and twice its standard error.

Dropping `Attr21` costs 0.0825 ± 0.0132 PR-AUC, and the effect shows on 24 of 25 folds. For
scale: the model's whole lift over random ranking is 0.747, so a single column out of 64 carries
about a ninth of it — more than the entire distance from Altman's 1983 formula to a fitted
logistic regression.

precision@top-3% moves by 0.0815 ± 0.0282. That metric steps in units of 1/26, so the change
amounts to roughly two of the twenty-six companies in a month's review queue.

In [4]:
cols_no21 = [c for c in feature_cols if c != "Attr21"]

show(compare_pipelines(boosting(feature_cols), boosting(cols_no21), X_train, y_train))

pr_auc          +0.0825 ± 0.0132   folds in favour: 24 / 25
precision_at_k  +0.0815 ± 0.0282   folds in favour: 19 / 25


## The same column in the linear branch

Here `Attr21` arrives median-imputed and without a flag, and a linear model has no way to isolate
a narrow group sitting in the middle of a distribution: it multiplies and sums.

The difference is 0.0067 ± 0.0045 — formally above zero, consistent across 21 of 25 folds, and
twelve times smaller than in boosting: about 3% of what this branch gains over the logistic
baseline. On precision@top-3% it does not separate from zero at all.

First evidence about mechanism. Whatever `Attr21` carries is something a tree can isolate and a
linear model cannot.

In [5]:
show(compare_pipelines(linear(feature_cols), linear(cols_no21), X_train, y_train))

pr_auc          +0.0067 ± 0.0045   folds in favour: 21 / 25
precision_at_k  +0.0108 ± 0.0169   folds in favour: 10 / 25


## Does filling the gap remove the effect?

If the effect lives in the missingness, imputing it should destroy it.

It does not: 0.802 against 0.734 with the column dropped, a difference of 0.0677 ± 0.0106.
Substituting one constant into 80 rows leaves them sharing a value that occurs nowhere else at
that density, and a tree separates them with a single split on it. The imputed constant becomes
the marker.

The imputation here happens outside the pipeline, so the median is taken over the whole training
split instead of per fold. That is not acceptable for a reported metric. It is enough for an
order of magnitude, and both sides of the comparison carry the same flaw.

In [6]:
X_med = X_train.copy()
X_med["Attr21"] = X_med["Attr21"].fillna(X_med["Attr21"].median())

res_med = cross_validate_model(boosting(feature_cols), X_med, y_train)
res_drop = cross_validate_model(boosting(cols_no21), X_train, y_train)

print(
    f"imputed {res_med['pr_auc'].mean():.3f}   dropped {res_drop['pr_auc'].mean():.3f}"
)
print(report(res_med["pr_auc"] - res_drop["pr_auc"]))

imputed 0.802   dropped 0.734
+0.0677 ± 0.0106


## The column on rows where it was never missing

Dropping the 80 rows removes the subgroup outright, and with it 78 of the 306 positives — so the
absolute numbers below are not comparable to anything above. Only the paired difference is.

On the remaining 4307 rows `Attr21` is worth 0.0054 ± 0.0111: nothing. Once the subgroup is gone
the column adds no information beyond the other 63.

**Decision: `Attr21` leaves the feature set in both branches.** The measured cost is 0.0825 PR-AUC
in boosting and 0.0067 in the linear branch, and none of it comes from the column as a financial
ratio. All of it comes from 80 rows whose origin cannot be established: either prior-year revenue
really was zero, or the prior-year statement was never collected for companies already known to
have failed. Nothing inside the file separates the two, and they behave in opposite ways on live
counterparties, who file a prior-year statement by virtue of still trading. A number that holds
only in retrospect is worse than a smaller number that does not depend on knowing the answer.

In [7]:
mask = X_train["Attr21"].notna()
X_sub, y_sub = X_train[mask], y_train[mask]
print(f"rows {mask.sum()}, positives {y_sub.sum()}")

diff = compare_pipelines(boosting(feature_cols), boosting(cols_no21), X_sub, y_sub)
print("pr_auc", report(diff["pr_auc"]))

rows 4307, positives 228
pr_auc +0.0054 ± 0.0111


## Do the missingness indicators earn their place in boosting?

The same mechanism raises the question: if the model reconstructs a missingness flag from the
column, six explicit ones may add nothing. Against that, an indicator is a feature in its own
right — it can be picked at the root, and two indicators can interact with each other. Native
NaN routing offers neither.

Both sides see the same 63 columns, so the six flags are the only difference; the bare pipeline
has no `ColumnTransformer` to select columns for it, hence the filtering on the input.

PR-AUC differs by -0.0004 ± 0.0006. Zero, resolved to about 0.001 — a precision the ±2σ rule,
which sets a threshold near 0.08 here, could never have reached. `folds in favour` reads low on
precision@top-3% for a mechanical reason: the metric is a count out of 26, so most folds differ
by exactly zero rather than by a sign.

The flags stay in the factory regardless. The measurement was taken on default hyperparameters,
and a constrained model — larger minimum leaf size, fewer leaves — routes NaN under different
conditions, so this is worth re-testing after tuning rather than baking in. Six columns that
provably do nothing are cheap; being able to re-run the comparison is not.

In [9]:
show(
    compare_pipelines(
        boosting(cols_no21),
        Pipeline([("model", LGBMClassifier(verbose=-1))]),
        X_train[cols_no21],
        y_train,
    )
)

pr_auc          -0.0004 ± 0.0006   folds in favour: 6 / 25
precision_at_k  +0.0015 ± 0.0031   folds in favour: 1 / 25


## What carries forward

`Attr21` is removed from the feature set, and the decision lives in `EXCLUDED_FEATURES` so that no
later experiment can quietly reintroduce it. The six missingness indicators stay in both branches
— measured useless in boosting, kept deliberately and on a stated condition.

Both findings, with their numbers, belong in README Limitations: the model reported here scores
0.734 rather than 0.817 because it declines to use a pattern it cannot vouch for.